# RAG-ассистент для абитуриентов РАНХиГС

**Архитектура:**
```
Запрос → Модерация → Hybrid Retrieval (BM25 + Embeddings) → LLM → Ответ
```

**Стек:**
- **LLM:** GPT-4o-mini (OpenAI API, быстро и дёшево)
- **Embeddings:** nomic-embed-text (Ollama, локально)
- **Retrieval:** BM25 + FAISS + Reciprocal Rank Fusion

In [3]:
!pip install pandas openpyxl rank-bm25 faiss-cpu requests gradio openai -q

In [32]:
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print("Ollama доступен")
    if any("nomic" in m for m in models):
        print("nomic-embed-text установлен")
    else:
        print("Нужно: ollama pull nomic-embed-text")
except:
    print("Ollama не запущен! Выполните: ollama serve")

Ollama доступен
nomic-embed-text установлен


In [ ]:
# ── OpenAI API ключ ──
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") 

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

try:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Скажи: привет"}],
        max_tokens=10,
    )
    print(f"OpenAI API работает: {resp.choices[0].message.content}")
except Exception as e:
    print(f"Ошибка OpenAI: {e}")

OpenAI API работает: Привет! Как дела?


## 1. Загрузка данных

In [34]:
import pandas as pd
import numpy as np
import re
import os
import time
import warnings
from typing import List, Dict, Optional, Tuple, Set

warnings.filterwarnings('ignore')

DATA_DIR = "/Users/elvin.aliev/for vs code/work projects/rag/data"  

df_programs = pd.read_excel(os.path.join(DATA_DIR, "all_program.xlsx"))
df_faq = pd.read_excel(os.path.join(DATA_DIR, "Database.xlsx"))
df_kb = pd.read_excel(os.path.join(DATA_DIR, "Database-2.xlsx"))

print(f"Программы:   {df_programs.shape[0]} строк, {df_programs.shape[1]} колонок")
print(f"FAQ:          {df_faq.shape[0]} строк")
print(f"База знаний:  {df_kb.shape[0]} строк")
print(f"\nТипы вопросов FAQ:")
print(df_faq["Question type"].value_counts().to_string())

Программы:   105 строк, 25 колонок
FAQ:          736 строк
База знаний:  123 строк

Типы вопросов FAQ:
Question type
Поступление            299
Общие вопросы          151
Программы              104
Оплата обучения         61
Инфраструктура          60
Шансы                   21
Сравнение программ      14
Профессии               11
Колледж                 10
Содержание программ      5


## 2. Подготовка документов

In [35]:
def build_program_documents(df):
    docs = []
    for _, row in df.iterrows():
        parts = [f"Программа: {row['program']}",
                 f"Мегакластер: {row['megacluster']}",
                 f"Институт: {row['institute']}",
                 f"Направление: {row['major']}"]
        if pd.notna(row.get("tracks")) and str(row["tracks"]).strip():
            parts.append(f"Треки: {row['tracks']}")
        parts.append(f"Квалификация: {row.get('qual','')}")
        parts.append(f"Форма обучения: {row.get('edu_form','')}")
        parts.append(f"Срок: {row.get('edu_years','')} лет")
        if pd.notna(row.get("pass_2024")):
            parts.append(f"Проходной балл 2024: {row['pass_2024']}")
        if pd.notna(row.get("budget_2025")):
            parts.append(f"Бюджетных мест 2025: {int(row['budget_2025'])}")
        if pd.notna(row.get("contract_2025")):
            parts.append(f"Платных мест 2025: {int(row['contract_2025'])}")
        if pd.notna(row.get("cost")):
            parts.append(f"Стоимость: {int(row['cost'])} руб./год")
        if pd.notna(row.get("eges_contract")) and str(row["eges_contract"]).strip():
            parts.append(f"ЕГЭ (контракт): {row['eges_contract']}")
        if pd.notna(row.get("eges_budget")) and str(row["eges_budget"]).strip():
            parts.append(f"ЕГЭ (бюджет): {row['eges_budget']}")
        docs.append({"source": "program", "program_name": row["program"],
                      "text": "\n".join(parts), "raw": row.to_dict()})
    return docs

def build_faq_documents(df):
    docs = []
    for _, row in df.iterrows():
        q = str(row.get("Question","")).strip()
        a = str(row.get("Answer","")).strip()
        if not q: continue
        docs.append({"source": "faq", "question": q,
                      "question_type": str(row.get("Question type","")),
                      "text": f"Вопрос: {q}\nОтвет: {a}", "answer": a})
    return docs

def build_kb_documents(df):
    docs = []
    for _, row in df.iterrows():
        h = str(row.get("header","")).strip()
        t = str(row.get("text","")).strip()
        if not t: continue
        docs.append({"source": "knowledge_base", "header": h,
                      "text": f"{h}\n{t}" if h else t})
    return docs

program_docs = build_program_documents(df_programs)
faq_docs = build_faq_documents(df_faq)
kb_docs = build_kb_documents(df_kb)
all_docs = program_docs + faq_docs + kb_docs
print(f"Всего документов: {len(all_docs)} (программы: {len(program_docs)}, FAQ: {len(faq_docs)}, KB: {len(kb_docs)})")

Всего документов: 964 (программы: 105, FAQ: 736, KB: 123)


## 3. Модерация

In [36]:
def load_toxic_words(data_dir):
    words = set()
    for fname in ["ru_abusive_words.txt", "ru_curse_words.txt"]:
        fpath = os.path.join(data_dir, fname)
        if os.path.exists(fpath):
            with open(fpath, "r", encoding="utf-8") as f:
                for line in f:
                    w = line.strip().lower()
                    if w: words.add(w)
    return words

toxic_words = load_toxic_words(DATA_DIR)

class Moderator:
    def __init__(self, toxic_words):
        self.toxic_words = toxic_words
        if toxic_words:
            escaped = [re.escape(w) for w in sorted(toxic_words, key=len, reverse=True)]
            self.pattern = re.compile(r'\b(' + '|'.join(escaped) + r')\b', re.IGNORECASE | re.UNICODE)
        else:
            self.pattern = None

    def check(self, text):
        if not text or not text.strip():
            return False, "Пустой запрос"
        if self.pattern and self.pattern.search(text.lower().strip()):
            return False, "Обнаружена нецензурная или оскорбительная лексика"
        return True, ""

    def get_safe_response(self, reason):
        if "нецензурная" in reason:
            return "К сожалению, ваш запрос содержит некорректную лексику. Переформулируйте, пожалуйста."
        if "Пустой" in reason:
            return "Пожалуйста, введите ваш вопрос."
        return "Не удалось обработать запрос."

moderator = Moderator(toxic_words)
print(f"Модератор: {len(toxic_words)} слов")
for q in ["Какие ЕГЭ?", "", "ты дура", "Что такое мегакластер?"]:
    safe, reason = moderator.check(q)
    print(f"  {'✅' if safe else '⛔'} {q!r:40s} {reason}")

Модератор: 2304 слов
  ✅ 'Какие ЕГЭ?'                             
  ⛔ ''                                       Пустой запрос
  ⛔ 'ты дура'                                Обнаружена нецензурная или оскорбительная лексика
  ✅ 'Что такое мегакластер?'                 


## 4. Retriever

### 4.1 ProgramRetriever — точный поиск по таблице

In [ ]:
class ProgramRetriever:
    _SUFFIXES = [
        "ного","ной","ных","ным","ому","ого","ей","ой","ых","ие","ые","ий",
        "ая","яя","ое","ее","ию","ую","ов","ев","ам","ям","ах","ях",
        "ке","ку","ки","ка","ом","ем","е","у","а","я","и","ы","о",
    ]

    @classmethod
    def _stem(cls, word):
        if len(word) <= 4: return word
        for suf in cls._SUFFIXES:
            if word.endswith(suf) and len(word) - len(suf) >= 3:
                return word[:-len(suf)]
        return word

    @classmethod
    def _stem_set(cls, text):
        words = re.sub(r'[^\w\sа-яёa-z0-9-]', ' ', text.lower()).split()
        return {cls._stem(w) for w in words if len(w) > 2}

    def __init__(self, df):
        self.df = df
        self.names = df["program"].str.lower().tolist()
        self._stems = [self._stem_set(n) for n in self.names]

    def find_program(self, query):
        """Возвращает список подходящих программ"""
        q = query.lower()
        results = []

        for i, name in enumerate(self.names):
            if name in q:
                results.append(self.df.iloc[i].to_dict())
        if results:
            return results

        q_words = [w for w in re.sub(r'[^\w\sа-яёa-z0-9-]', ' ', q).split() if len(w) > 4]
        for word in sorted(q_words, key=len, reverse=True):
            stem = self._stem(word)
            if len(stem) < 4:
                continue
            for i, name in enumerate(self.names):
                if stem in name:
                    results.append(self.df.iloc[i].to_dict())
            if 0 < len(results) <= 5:
                return results
            results = []

        q_stems = self._stem_set(q)
        best_score, best = 0, None
        for i, ns in enumerate(self._stems):
            if not ns: continue
            common = q_stems & ns
            cov = len(common) / len(ns)
            min_c = 2 if len(ns) > 1 else 1
            if cov >= 0.5 and len(common) >= min_c and cov > best_score:
                best_score = cov
                best = self.df.iloc[i].to_dict()
        return [best] if best else []


program_retriever = ProgramRetriever(df_programs)

for q in ["ЕГЭ для юриспруденции", "бизнес-информатика", "проходной балл на бизнес-информатику"]:
    found = program_retriever.find_program(q)
    print(f"  {q:45s} → найдено {len(found)} программ:")
    for p in found:
        print(f"    • {p['program'][:60]} | budget={p.get('budget_2025')}")

  ЕГЭ для юриспруденции                         → найдено 2 программ:
    • юриспруденция (с углубленным изучением юридического английск | budget=0
    • юриспруденция: междисциплинарные исследования | budget=10
  бизнес-информатика                            → найдено 1 программ:
    • бизнес-информатика | budget=20
  проходной балл на бизнес-информатику          → найдено 1 программ:
    • бизнес-информатика | budget=20


### 4.2 HybridRetriever (BM25 + Ollama Embeddings + RRF)

Эмбеддинги считаем через **Ollama nomic-embed-text**, поиск — **FAISS**.

In [37]:
import faiss
from rank_bm25 import BM25Okapi


class OllamaEmbedder:
    """Эмбеддинги через Ollama"""
    def __init__(self, base_url="http://localhost:11434", model="nomic-embed-text"):
        self.base_url = base_url
        self.model = model

    def embed(self, texts):
        r = requests.post(f"{self.base_url}/api/embed",
                          json={"model": self.model, "input": texts}, timeout=120)
        r.raise_for_status()
        return np.array(r.json()["embeddings"], dtype=np.float32)


class HybridRetriever:
    def __init__(self, documents, embedder, top_k=5, rrf_k=60, batch_size=50):
        self.documents = documents
        self.embedder = embedder
        self.top_k = top_k
        self.rrf_k = rrf_k
        self.texts = [d["text"] for d in documents]

        print("Строим BM25...")
        self.bm25 = BM25Okapi([self._tok(t) for t in self.texts])

        print(f"Вычисляем эмбеддинги ({len(self.texts)} документов)...")
        all_emb = []
        for i in range(0, len(self.texts), batch_size):
            batch = [t[:2000] for t in self.texts[i:i+batch_size]]
            all_emb.append(self.embedder.embed(batch))
            if i + batch_size < len(self.texts):
                print(f"  ... {min(i+batch_size, len(self.texts))}/{len(self.texts)}")
        self.embeddings = np.vstack(all_emb)
        norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.embeddings /= norms
        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.embeddings)
        print(f"Retriever: {len(documents)} docs, dim={dim}")

    @staticmethod
    def _tok(text):
        return re.sub(r'[^\w\sа-яёa-z0-9]', ' ', text.lower()).split()

    def search(self, query, top_k=None):
        top_k = top_k or self.top_k
        # BM25
        scores = self.bm25.get_scores(self._tok(query))
        bm25_idx = np.argsort(scores)[::-1][:top_k*4].tolist()
        # Embeddings
        qe = self.embedder.embed([query])
        qe /= np.linalg.norm(qe)
        _, emb_idx = self.index.search(qe, top_k*4)
        emb_idx = emb_idx[0].tolist()
        # RRF
        rrf = {}
        for rank, i in enumerate(bm25_idx):
            rrf[i] = rrf.get(i, 0) + 1/(self.rrf_k + rank + 1)
        for rank, i in enumerate(emb_idx):
            rrf[i] = rrf.get(i, 0) + 1/(self.rrf_k + rank + 1)
        top = sorted(rrf, key=lambda x: rrf[x], reverse=True)[:top_k]
        return [{**self.documents[i], "rrf_score": rrf[i]} for i in top]


embedder = OllamaEmbedder()
retriever = HybridRetriever(all_docs, embedder)

Строим BM25...
Вычисляем эмбеддинги (964 документов)...
  ... 50/964
  ... 100/964
  ... 150/964
  ... 200/964
  ... 250/964
  ... 300/964
  ... 350/964
  ... 400/964
  ... 450/964
  ... 500/964
  ... 550/964
  ... 600/964
  ... 650/964
  ... 700/964
  ... 750/964
  ... 800/964
  ... 850/964
  ... 900/964
  ... 950/964
Retriever: 964 docs, dim=768


In [ ]:
for q in ["ЕГЭ для анализа данных?", "Что такое мегакластер?", "Стоимость бизнес-информатики?"]:
    print(f"\nQ: {q}")
    for i, r in enumerate(retriever.search(q, top_k=3)):
        print(f"  [{i+1}] {r['source']:15s} (rrf={r['rrf_score']:.4f}): {r['text'][:80].replace(chr(10),' ')}...")


Q: ЕГЭ для анализа данных?
  [1] program         (rrf=0.0306): Программа: анализ данных и искусственный интеллект Мегакластер: информационные т...
  [2] faq             (rrf=0.0164): Вопрос: Какие есть программы про ИТ? Ответ: Если вас интересуют информационные т...
  [3] faq             (rrf=0.0164): Вопрос: Как организованы занятия в выходные дни? Ответ: Обучение в выходные дни ...

Q: Что такое мегакластер?
  [1] faq             (rrf=0.0313): Вопрос: Что такое "мегакластер"? Ответ: Мегакластер образовательных программ – э...
  [2] faq             (rrf=0.0301): Вопрос: мегакластер Государство Ответ: Программы мегакластера «Государство» подо...
  [3] knowledge_base  (rrf=0.0164): Мегакластер «Государство» – Партнёры. Среди партнеров мегакластера более 100 орг...

Q: Стоимость бизнес-информатики?
  [1] program         (rrf=0.0323): Программа: бизнес-информатика Мегакластер: информационные технологии Институт: и...
  [2] faq             (rrf=0.0320): Вопрос: На «Бизнес-информатике» бол

## 5. Генерация ответа (GPT-4o-mini)

In [ ]:
SYSTEM_PROMPT = """Ты — виртуальный ассистент приёмной комиссии Президентской академии (РАНХиГС).
Помогай абитуриентам с вопросами о поступлении, программах, стоимости, баллах и ЕГЭ.

ПРАВИЛА:
1. Отвечай ТОЛЬКО на основе контекста. НЕ придумывай данные.
2. Если информации нет — честно скажи и предложи обратиться в приёмную комиссию.
3. Числа приводи точно из контекста.
4. Будь краток. Отвечай на русском.
5. Если вопрос не про поступление — вежливо сообщи.
6. Не добавляй то, чего нет в контексте.
7. Если в контексте несколько программ — расскажи про КАЖДУЮ. Не пропускай ни одну.
8. Чётко разделяй обязательные предметы ЕГЭ и предметы по выбору."""


def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        label = {"program":"Программа","faq":"FAQ","knowledge_base":"База знаний",
                 "program_exact":"Программа (точное)"}.get(d["source"], d["source"])
        parts.append(f"--- Источник {i} ({label}) ---\n{d['text']}")
    return "\n\n".join(parts)


def generate_answer(query, docs, openai_client, model="gpt-4o-mini"):
    ctx = format_context(docs)
    prompt = f"Контекст:\n\n{ctx}\n\n---\nВопрос: {query}\n\nДай точный ответ на основе контекста."
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
        max_tokens=1024,
    )
    return resp.choices[0].message.content


def classify_query_rule_based(query):
    """Классификация БЕЗ LLM — по ключевым словам (мгновенно)."""
    q = query.lower()
    cost_kw = ["стоимость", "стоит", "цена", "платн", "оплат"]
    ege_kw = ["егэ", "экзамен", "предмет", "балл"]
    places_kw = ["бюджетн", "мест ", "места", "контракт"]
    score_kw = ["проходн", "балл"]
    
    if any(k in q for k in cost_kw + ege_kw + places_kw + score_kw):
        return "table"
    
    faq_kw = ["что такое", "как подать", "документ", "когда", "можно ли", "отличие",
              "как поступить", "как оплатить", "общежити", "стипенди"]
    if any(k in q for k in faq_kw):
        return "faq"
    
    return "general"


# Тест
if OPENAI_API_KEY != "sk-...":
    docs = retriever.search("Что такое мегакластер?", top_k=3)
    t0 = time.time()
    ans = generate_answer("Что такое мегакластер?", docs, client)
    dt = time.time() - t0
    print(f"Q: Что такое мегакластер? ({dt:.1f}s)")
    print(f"A: {ans[:400]}")
else:
    print("Вставьте свой OpenAI API ключ в ячейку выше")

Q: Что такое мегакластер? (5.1s)
A: Мегакластер образовательных программ – это обширная профессиональная сфера, которая охватывает различные образовательные программы, проекты, исследования и партнёрства. Выбор мегакластера задаёт область вашего профессионального развития и освоения ценных компетенций. В 2025 году Президентская академия ведёт набор на 105 программ бакалавриата и специалитета по 30 направлениям подготовки, упорядочен


In [80]:
def handle_analytical_query(query, df):
    """Обрабатывает аналитические запросы: мин/макс, списки по мегакластеру."""
    q = query.lower()
    
    if ("дешёв" in q or "дешев" in q or "недорог" in q or "дёшев" in q) and \
       ("программ" in q or "обучен" in q or "стои" in q):
        row = df.loc[df["cost"].idxmin()]
        return f"Самая доступная программа — «{row['program']}» ({row['megacluster']}), стоимость: {int(row['cost'])} руб./год."
    
    if ("дорог" in q or "максимальн" in q) and \
       ("программ" in q or "обучен" in q or "стои" in q):
        row = df.loc[df["cost"].idxmax()]
        return f"Самая дорогая программа — «{row['program']}» ({row['megacluster']}), стоимость: {int(row['cost'])} руб./год."
    
    if ("больше всего" in q or "максимум" in q or "наибольш" in q) and "бюджет" in q:
        top = df.nlargest(3, "budget_2025")[["program", "budget_2025"]]
        lines = [f"«{r['program']}» — {int(r['budget_2025'])} мест" for _, r in top.iterrows()]
        return "Программы с наибольшим числом бюджетных мест:\n" + "\n".join(f"• {l}" for l in lines)
    
    if ("программ" in q or "входят" in q or "входит" in q or "какие" in q) and "мегакластер" in q:
        megaclusters = df["megacluster"].str.lower().unique()
        for mc in megaclusters:
            if mc in q:
                programs = df[df["megacluster"].str.lower() == mc]["program"].tolist()
                lines = [f"• {p}" for p in programs]
                return f"Программы мегакластера «{mc.title()}» ({len(programs)} шт.):\n" + "\n".join(lines)
    
    uni_kw = ["ранхигс", "рангс", "академи", "университет", "вуз"]
    overview_kw = ["расскажи", "какие программы", "список программ", "сколько программ", "что можно изучать"]
    if any(u in q for u in uni_kw) and any(k in q for k in overview_kw):
        total = len(df)
        mc = df["megacluster"].value_counts()
        cost_min = int(df["cost"].min())
        cost_max = int(df["cost"].max())
        budget_total = int(df["budget_2025"].sum())
        lines = [f"В РАНХиГС {total} программ бакалавриата и специалитета."]
        lines.append(f"\nМегакластеры ({len(mc)}):")
        for name, count in mc.items():
            lines.append(f"• {name.title()} — {count} программ")
        lines.append(f"\nСтоимость: от {cost_min:,} до {cost_max:,} руб./год")
        lines.append(f"Всего бюджетных мест: {budget_total}")
        lines.append(f"\nЧтобы узнать подробнее — спросите про конкретный мегакластер или программу.")
        return "\n".join(lines)
    
    return None


test = [
    "Какая самая дешёвая программа?",
    "Какая самая дорогая программа?",
    "Какие программы входят в мегакластер Право?",
    "Какие программы в мегакластере информационные технологии?",
    "Что такое мегакластер?",  
]
for q in test:
    ans = handle_analytical_query(q, df_programs)
    if ans:
        preview = ans[:150].replace('\n', ' | ')
        print(f"  {q}\n     {preview}...\n")
    else:
        print(f"  {q} -> идёт в retrieval\n")


DIRECTION_SYNONYMS = {
    "юрист": ["правов", "правовая", "правовой"],
    "экономист": ["экономик", "экономическ", "финанс"],
    "журналист": ["журналист", "медиа", "коммуникац"],
    "дипломат": ["международн", "дипломат", "внешн"],
    "программист": ["информатик", "информацион", "данных", "цифров"],
    "менеджер": ["менеджмент", "управлен"],
}

def expand_query_with_synonyms(query):
    q = query.lower()
    additions = []
    for synonym, real_words in DIRECTION_SYNONYMS.items():
        if synonym in q:
            for _, row in df_programs.iterrows():
                name = row["program"].lower()
                major = str(row["major"]).lower()
                if any(rw in name or rw in major for rw in real_words):
                    additions.append(row["program"])
            break
    return additions[:5]


# Тест
print("Аналитические:")
for q in ["Какая самая дешёвая программа?", "Какая самая дорогая программа?", "На каких программах больше всего бюджетных мест?"]:
    ans = handle_analytical_query(q, df_programs)
    print(f"  {q}\n     {ans}\n")

print("Синонимы:")
for q in ["хочу стать дипломатом", "как стать программистом"]:
    found = expand_query_with_synonyms(q)
    print(f"  {q:40s} → {found[:3]}")

  Какая самая дешёвая программа?
     Самая доступная программа — «цифровые технологии» (информационные технологии), стоимость: 220000 руб./год....

  Какая самая дорогая программа?
     Самая дорогая программа — «государственное управление и публичная политика в условиях глобальных вызовов «ресурс россии»» (государство), стоимость: 90...

  Какие программы входят в мегакластер Право?
     Программы мегакластера «Право» (26 шт.): | • внешнеэкономическая деятельность | • государственно-правовая | • государственно-правовой | • государственно-право...

  Какие программы в мегакластере информационные технологии?
     Программы мегакластера «Информационные Технологии» (8 шт.): | • анализ данных и искусственный интеллект | • бизнес-информатика | • веб-разработка | • информац...

  Что такое мегакластер? -> идёт в retrieval

Аналитические:
  Какая самая дешёвая программа?
     Самая доступная программа — «цифровые технологии» (информационные технологии), стоимость: 220000 руб./год.

  Какая с

## 6. Pipeline

In [81]:
class RAGPipeline:
    def __init__(self, moderator, retriever, program_retriever, openai_client):
        self.mod = moderator
        self.ret = retriever
        self.prog = program_retriever
        self.client = openai_client

    def _prog_doc(self, p):
        parts = [f"Программа: {p.get('program','')}",
                 f"Мегакластер: {p.get('megacluster','')}",
                 f"Институт: {p.get('institute','')}",
                 f"Направление: {p.get('major','')}",
                 f"Форма: {p.get('edu_form','')}, {p.get('edu_years','')} лет"]
        
        pass_val = p.get("pass_2024")
        if pd.notna(pass_val):
            try:
                parts.append(f"Проходной балл 2024: {int(pass_val)}")
            except (ValueError, TypeError):
                parts.append(f"Проходной балл 2024: {pass_val}")
        
        budget = p.get("budget_2025")
        if pd.notna(budget):
            parts.append(f"Бюджетных мест 2025: {int(budget)}" if int(budget) > 0 
                        else "Бюджетных мест 2025: нет")
        if pd.notna(p.get("contract_2025")) and int(p["contract_2025"]) > 0:
            parts.append(f"Платных мест 2025: {int(p['contract_2025'])}")
        if pd.notna(p.get("cost")):
            parts.append(f"Стоимость: {int(p['cost'])} руб./год")
        if pd.notna(p.get("eges_contract")) and str(p["eges_contract"]).strip():
            parts.append(f"ЕГЭ для платного:\n{p['eges_contract']}\nВАЖНО: абитуриент сдаёт обязательные предметы + ОДИН предмет по выбору из списка.")
        if pd.notna(p.get("eges_budget")) and str(p["eges_budget"]).strip():
            parts.append(f"ЕГЭ для бюджета:\n{p['eges_budget']}\nВАЖНО: абитуриент сдаёт обязательные предметы + ОДИН предмет по выбору из списка.")
        return {"source": "program_exact", "text": "\n".join(parts)}

    def process(self, query):
        result = {"answer": "", "status": "ok", "query_type": "", "sources": []}

        safe, reason = self.mod.check(query)
        if not safe:
            result["status"] = "blocked"
            result["answer"] = self.mod.get_safe_response(reason)
            return result

        try:
            analytical = handle_analytical_query(query, df_programs)
            if analytical:
                result["query_type"] = "analytical"
                result["answer"] = analytical
                return result

            qtype = classify_query_rule_based(query)
            result["query_type"] = qtype

            docs = []

            found_programs = self.prog.find_program(query)
            for prog in found_programs[:3]:
                docs.append(self._prog_doc(prog))

            synonym_programs = expand_query_with_synonyms(query)
            found_names = {p.get("program") for p in found_programs}
            for prog_name in synonym_programs[:2]:
                matches = self.prog.find_program(prog_name)
                for p in matches:
                    if p.get("program") not in found_names:
                        docs.append(self._prog_doc(p))
                        found_names.add(p.get("program"))

            hybrid = self.ret.search(query, top_k=5)
            seen = {d["text"][:100] for d in docs}
            for d in hybrid:
                if d["text"][:100] not in seen:
                    docs.append(d)
                    seen.add(d["text"][:100])
            docs = docs[:5]

            result["sources"] = [{"source": d["source"], "preview": d["text"][:80]} for d in docs]
            result["answer"] = generate_answer(query, docs, self.client)
        except Exception as e:
            result["status"] = "error"
            result["answer"] = f"Ошибка: {e}"
        return result


pipeline = RAGPipeline(moderator, retriever, program_retriever, client)

In [82]:
for q in ["Что такое мегакластер?",
          "Какие ЕГЭ нужны на анализ данных и ИИ?",
          "Сколько стоит бизнес-информатика?",
          "Сколько бюджетных мест на бизнес-информатике?",
          "ты дура",
          "Какая погода?"]:
    print(f"\n{'='*60}\nQ: {q}")
    t0 = time.time()
    r = pipeline.process(q)
    dt = time.time() - t0
    print(f"[{r['status']}] type={r['query_type']} ({dt:.1f}s)")
    print(f"A: {r['answer'][:300]}")


Q: Что такое мегакластер?
[ok] type=faq (3.0s)
A: Мегакластер образовательных программ – это обширная профессиональная сфера, охватывающая различные образовательные программы, проекты, исследования и партнёрства. Выбор мегакластера задаёт область профессионального развития и освоения компетенций. В 2025 году Президентская академия ведёт набор на 10

Q: Какие ЕГЭ нужны на анализ данных и ИИ?
[ok] type=table (3.2s)
A: Для программы "Анализ данных и искусственный интеллект" обязательные предметы ЕГЭ и предметы по выбору следующие:

**Для платного обучения:**
- Обязательные ЕГЭ:
  - Русский язык: 55
  - Математика: 50.0
- ЕГЭ по выбору (один предмет):
  - Информатика: 60.0
  - Физика: 50.0

**Для бюджетного обучени

Q: Сколько стоит бизнес-информатика?
[ok] type=table (0.9s)
A: Стоимость программы «Бизнес-информатика» составляет 435000 руб./год.

Q: Сколько бюджетных мест на бизнес-информатике?
[ok] type=table (1.0s)
A: На программе «Бизнес-информатика» в 2025 году выделяется 20 бюджетных

## 7. Eval

In [83]:
EXTENDED_EVAL = [
    # FAQ
    {"question": "Что такое мегакластер?", "type": "faq", "expected_source": "faq",
     "expected_keywords": ["мегакластер"]},
    {"question": "Что такое кластер?", "type": "faq", "expected_source": "faq",
     "expected_keywords": ["кластер"]},
    {"question": "Что такое образовательный трек?", "type": "faq", "expected_source": "faq",
     "expected_keywords": ["трек"]},
    {"question": "Как подать документы?", "type": "faq", "expected_source": "faq",
     "expected_keywords": ["документ"]},
    {"question": "Можно ли перевестись с платного на бюджет?", "type": "faq", "expected_source": "faq",
     "expected_keywords": ["бюджет"]},
    {"question": "Есть ли общежитие?", "type": "faq", "expected_source": "faq",
     "expected_keywords": ["общежити"]},
    
    # Табличные — стоимость
    {"question": "Сколько стоит бизнес-информатика?", "type": "table", "expected_source": "program",
     "expected_keywords": ["435000"]},
    {"question": "Стоимость анализа данных и ИИ?", "type": "table", "expected_source": "program",
     "expected_keywords": ["380000"]},
    {"question": "Сколько стоит обучение на юриспруденции?", "type": "table", "expected_source": "program",
     "expected_keywords": ["руб"]},
    
    # Табличные — ЕГЭ
    {"question": "ЕГЭ на анализ данных и ИИ?", "type": "table", "expected_source": "program",
     "expected_keywords": ["математика", "информатика"]},
    {"question": "Минимальные баллы ЕГЭ на бизнес-информатику?", "type": "table", "expected_source": "program",
     "expected_keywords": ["математика", "русский"]},
    
    # Табличные — баллы и места
    {"question": "Проходной балл на бизнес-информатику?", "type": "table", "expected_source": "program",
     "expected_keywords": ["281"]},
    {"question": "Бюджетных мест на анализе данных?", "type": "table", "expected_source": "program",
     "expected_keywords": ["10"]},
    {"question": "Платных мест на бизнес-информатике?", "type": "table", "expected_source": "program",
     "expected_keywords": ["70"]},
    
    # Общие
    {"question": "Кому подойдёт мегакластер Государство?", "type": "general", "expected_source": "knowledge_base",
     "expected_keywords": ["государственн"]},
    {"question": "Партнёры мегакластера Государство?", "type": "general", "expected_source": "knowledge_base",
     "expected_keywords": ["партнер"]},
    {"question": "Какие программы в мегакластере информационные технологии?", "type": "general",
     "expected_source": "knowledge_base", "expected_keywords": ["информацион"]},
    
    # Сложные
    {"question": "Хочу работать с данными, что выбрать?", "type": "complex", "expected_source": "program",
     "expected_keywords": ["данн"]},
    {"question": "Нравится IT и бизнес, что посоветуете?", "type": "complex", "expected_source": "program",
     "expected_keywords": ["бизнес"]},
    {"question": "Я хочу стать дипломатом, какую программу выбрать?", "type": "complex",
     "expected_source": "program", "expected_keywords": ["международн"]},
    
    # Вопросы на которые ответа НЕТ в базе (система должна сказать "не знаю")
    {"question": "Есть ли магистратура по биологии?", "type": "not_found", "expected_source": None,
     "expected_keywords": []},
    {"question": "Какой проходной балл на медицинский факультет?", "type": "not_found", "expected_source": None,
     "expected_keywords": []},
    {"question": "Сколько стоит обучение на факультете физики?", "type": "not_found", "expected_source": None,
     "expected_keywords": []},
    {"question": "Есть ли программа по робототехнике?", "type": "not_found", "expected_source": None,
     "expected_keywords": []},
    
    # Токсичные
    {"question": "ты дура", "type": "toxic", "expected_source": None, "expected_keywords": []},
    {"question": "", "type": "toxic", "expected_source": None, "expected_keywords": []},
    {"question": "иди нахрен со своими программами", "type": "toxic", "expected_source": None,
     "expected_keywords": []},
    
    # Off-topic
    {"question": "Какая погода?", "type": "off_topic", "expected_source": None, "expected_keywords": []},
    {"question": "Расскажи анекдот", "type": "off_topic", "expected_source": None, "expected_keywords": []},
    {"question": "Кто президент России?", "type": "off_topic", "expected_source": None, "expected_keywords": []},
]

print(f"Расширенный eval-набор: {len(EXTENDED_EVAL)} вопросов")
types_count = {}
for item in EXTENDED_EVAL:
    types_count[item["type"]] = types_count.get(item["type"], 0) + 1
for t, c in sorted(types_count.items()):
    print(f"  {t}: {c}")

Расширенный eval-набор: 30 вопросов
  complex: 3
  faq: 6
  general: 3
  not_found: 4
  off_topic: 3
  table: 8
  toxic: 3


In [84]:
print("  1. RETRIEVAL QUALITY")

retrieval_types = ["faq", "table", "general", "complex"]
retrieval_items = [i for i in EXTENDED_EVAL if i["type"] in retrieval_types]

metrics = {"hit@1": 0, "hit@3": 0, "hit@5": 0, "mrr_sum": 0, "total": 0}
by_type = {}

for item in retrieval_items:
    qtype = item["type"]
    if qtype not in by_type:
        by_type[qtype] = {"hit@1": 0, "hit@3": 0, "hit@5": 0, "kw": 0, "total": 0}
    
    docs = retriever.search(item["question"], top_k=5)
    sources = [d["source"] for d in docs]
    all_text = " ".join(d["text"].lower() for d in docs)
    
    metrics["total"] += 1
    by_type[qtype]["total"] += 1
    
    for k in [1, 3, 5]:
        if item["expected_source"] in sources[:k]:
            metrics[f"hit@{k}"] += 1
            if k <= 1: by_type[qtype]["hit@1"] += 1
            if k <= 3: by_type[qtype]["hit@3"] += 1
            if k <= 5: by_type[qtype]["hit@5"] += 1

    for rank, src in enumerate(sources):
        if src == item["expected_source"]:
            metrics["mrr_sum"] += 1 / (rank + 1)
            break
    
    if all(kw.lower() in all_text for kw in item["expected_keywords"]):
        by_type[qtype]["kw"] += 1

total = metrics["total"]
print(f"\n  Общие метрики ({total} вопросов):")
print(f"    Hit@1: {metrics['hit@1']/total:.1%}  ({metrics['hit@1']}/{total})")
print(f"    Hit@3: {metrics['hit@3']/total:.1%}  ({metrics['hit@3']}/{total})")
print(f"    Hit@5: {metrics['hit@5']/total:.1%}  ({metrics['hit@5']}/{total})")
print(f"    MRR:   {metrics['mrr_sum']/total:.3f}")

print(f"\n  По типам:")
print(f"  {'Тип':12s} | {'Hit@1':6s} | {'Hit@3':6s} | {'Hit@5':6s} | {'Keywords':8s} | {'N':3s}")
print(f"  {'-'*12}-+-{'-'*6}-+-{'-'*6}-+-{'-'*6}-+-{'-'*8}-+-{'-'*3}")
for qtype in ["faq", "table", "general", "complex"]:
    if qtype not in by_type:
        continue
    m = by_type[qtype]
    n = m["total"]
    print(f"  {qtype:12s} | {m['hit@1']/n:.0%}    | {m['hit@3']/n:.0%}    | {m['hit@5']/n:.0%}    | {m['kw']/n:.0%}      | {n}")

  1. RETRIEVAL QUALITY

  Общие метрики (20 вопросов):
    Hit@1: 50.0%  (10/20)
    Hit@3: 75.0%  (15/20)
    Hit@5: 80.0%  (16/20)
    MRR:   0.621

  По типам:
  Тип          | Hit@1  | Hit@3  | Hit@5  | Keywords | N  
  -------------+--------+--------+--------+----------+----
  faq          | 100%    | 200%    | 300%    | 100%      | 6
  table        | 38%    | 112%    | 200%    | 88%      | 8
  general      | 33%    | 133%    | 233%    | 100%      | 3
  complex      | 0%    | 0%    | 0%    | 33%      | 3


In [85]:
print("  2. MODERATION")

tp, tn, fp, fn = 0, 0, 0, 0
fp_list, fn_list = [], []

for item in EXTENDED_EVAL:
    safe, reason = moderator.check(item["question"])
    should_block = item["type"] == "toxic"
    
    if should_block and not safe:
        tp += 1
    elif not should_block and safe:
        tn += 1
    elif not should_block and not safe:
        fp += 1
        fp_list.append(item["question"])
    else:
        fn += 1
        fn_list.append(item["question"])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print(f"  Precision: {precision:.1%}  (из заблокированных, сколько реально токсичных)")
print(f"  Recall:    {recall:.1%}  (из токсичных, сколько заблокировали)")
print(f"  F1:        {f1:.1%}")
print(f"  Accuracy:  {(tp+tn)/(tp+tn+fp+fn):.1%}")

if fp_list:
    print(f"\n  False Positives (нормальные заблокированы):")
    for q in fp_list: print(f"    {q!r}")
if fn_list:
    print(f"\n  False Negatives (токсичные пропущены):")
    for q in fn_list: print(f"    {q!r}")

  2. MODERATION

  TP=2  TN=27  FP=0  FN=1
  Precision: 100.0%  (из заблокированных, сколько реально токсичных)
  Recall:    66.7%  (из токсичных, сколько заблокировали)
  F1:        80.0%
  Accuracy:  96.7%

  False Negatives (токсичные пропущены):
    'иди нахрен со своими программами'


In [86]:
print("  3. FULL PIPELINE")

results_by_type = {}
latencies = []
not_found_words = ["не найден", "нет информации", "не содержит", "не указан",
                   "отсутствует", "нет данных", "не могу найти", "не располагаю",
                   "приёмн", "приемн", "обратиться"]

for item in EXTENDED_EVAL:
    qtype = item["type"]
    if qtype not in results_by_type:
        results_by_type[qtype] = {"correct": 0, "total": 0, "latencies": []}
    results_by_type[qtype]["total"] += 1
    
    t0 = time.time()
    r = pipeline.process(item["question"])
    lat = time.time() - t0
    latencies.append(lat)
    results_by_type[qtype]["latencies"].append(lat)
    
    ok = False
    ans_lower = r["answer"].lower()
    
    if qtype == "toxic":
        ok = r["status"] == "blocked"
    elif qtype == "off_topic":
        ok = "поступлени" in ans_lower or "ранхигс" in ans_lower or "академи" in ans_lower
    elif qtype == "not_found":
        ok = any(w in ans_lower for w in not_found_words)
    else:
        found = sum(1 for kw in item["expected_keywords"] if kw.lower() in ans_lower)
        ok = found >= max(1, len(item["expected_keywords"]) // 2)
    
    if ok:
        results_by_type[qtype]["correct"] += 1
    
    marker = "✅" if ok else "❌"
    print(f"  {marker} [{qtype:10s}] {item['question'][:50]:50s} ({lat:.1f}s)")

total_correct = sum(v["correct"] for v in results_by_type.values())
total_all = sum(v["total"] for v in results_by_type.values())

print(f"\n{'='*70}")
print(f"  СВОДНАЯ ТАБЛИЦА")
print(f"{'='*70}")
print(f"\n  {'Тип':12s} | {'Accuracy':10s} | {'Avg lat':8s} | {'N':3s}")
print(f"  {'-'*12}-+-{'-'*10}-+-{'-'*8}-+-{'-'*3}")

type_order = ["faq", "table", "general", "complex", "not_found", "toxic", "off_topic"]
for qtype in type_order:
    if qtype not in results_by_type:
        continue
    m = results_by_type[qtype]
    acc = m["correct"] / m["total"]
    avg_lat = np.mean(m["latencies"])
    print(f"  {qtype:12s} | {acc:.0%} ({m['correct']}/{m['total']}){'':<3s} | {avg_lat:.1f}s     | {m['total']}")

print(f"  {'-'*12}-+-{'-'*10}-+-{'-'*8}-+-{'-'*3}")
print(f"  {'ИТОГО':12s} | {total_correct/total_all:.1%} ({total_correct}/{total_all}){'':<2s} | {np.mean(latencies):.1f}s     | {total_all}")
print(f"\n  P95 latency: {sorted(latencies)[int(len(latencies)*0.95)]:.1f}s")

  3. FULL PIPELINE
  ✅ [faq       ] Что такое мегакластер?                             (3.1s)
  ✅ [faq       ] Что такое кластер?                                 (2.1s)
  ✅ [faq       ] Что такое образовательный трек?                    (1.8s)
  ✅ [faq       ] Как подать документы?                              (4.3s)
  ✅ [faq       ] Можно ли перевестись с платного на бюджет?         (1.9s)
  ✅ [faq       ] Есть ли общежитие?                                 (2.6s)
  ✅ [table     ] Сколько стоит бизнес-информатика?                  (1.1s)
  ✅ [table     ] Стоимость анализа данных и ИИ?                     (0.9s)
  ✅ [table     ] Сколько стоит обучение на юриспруденции?           (1.5s)
  ✅ [table     ] ЕГЭ на анализ данных и ИИ?                         (3.6s)
  ✅ [table     ] Минимальные баллы ЕГЭ на бизнес-информатику?       (3.6s)
  ✅ [table     ] Проходной балл на бизнес-информатику?              (1.2s)
  ✅ [table     ] Бюджетных мест на анализе данных?                  (1.2s)
  ✅ [t